# Open-weight VLM roster benchmark (Colab)

Runs **6 locally-loaded open-weight VLMs** (not APIs) over the same V4a (synthetic tag) and
V4b (real chest tube) crops, under the *exact same blind prompt*, one model at a time with
full VRAM cleanup between models -- no runtime restart needed.

**Roster** (medical specialists first, then general VLMs):
1. `StanfordAIMI/CheXagent-2-3b`
2. `StanfordAIMI/CheXagent-8b`
3. `google/medgemma-4b-it`
4. `Qwen/Qwen2.5-VL-3B-Instruct`
5. `microsoft/Phi-3.5-vision-instruct`
6. `HuggingFaceTB/SmolVLM2-2.2B-Instruct`

**Version constraint (learned from the failed 2026-07-16 run):** `transformers` is pinned
`>=4.50,<5` and the bound at BOTH ends is load-bearing.
- **`<5`** because CheXagent-2-3b/8b's `trust_remote_code=True` code imports
  `is_tf_available`, which transformers 5.0 removed along with the whole TF/Flax backend.
  Colab's stock transformers is now 5.x, so this must be pinned down.
- **`>=4.50`** because MedGemma (Gemma 3) requires it.

`src/auditor/discovery/colab/chexagent_naming.ipynb` pins `==4.40.0` for CheXagent and reads
as though 4.40 specifically were required. The 2026-07-16 evidence says otherwise: the
failure there was `is_tf_available`, which exists throughout 4.x and was only dropped in 5.0
-- so any 4.x should do, and CheXagent is expected to coexist with the rest of the roster
under this pin. **This is a prediction, not yet an observed pass** -- CheXagent has never
actually loaded under 4.50 here. If it fails anyway, that's caught and logged, not fatal.

**If `transformers` was already imported in this session** (a previous run, or Colab's boot),
`!pip install` changes only what is on disk -- the old version stays live in memory and the
pin silently does nothing. The environment-guard cell below asserts the version to catch
exactly that; if it fires, `Runtime -> Restart session` and re-run from cell 1.

**This notebook does inference ONLY.** It writes `outputs/predictions_<model>_<dataset>.jsonl`
(one JSON object per line: `{"id", "kind", "description"}`) -- the same schema
`src/auditor/discovery/score_naming.py` already reads. Score locally after downloading:

```
python -m auditor.discovery score --manifest manifest_v4a.json \
    --predictions predictions_<model>_v4a.jsonl --confound tag
python -m auditor.discovery score --manifest manifest_v4b.json \
    --predictions predictions_<model>_v4b.jsonl --confound tube
```

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

**`google/medgemma-4b-it` is GATED** -- you must accept its license at
https://huggingface.co/google/medgemma-4b-it (log in, click "Agree and access repository")
and provide an HF token with read access in the login cell below, or it will fail-and-skip
like any other unavailable model.

**Upload, zipped from your local checkout, matching names:**
- `crops_v4a.zip` (from `experiments/v4a_naming/<run>/crops/`) + `manifest_v4a.json`
  (renamed copy of that run's `manifest.json`)
- `crops_v4b.zip` (from `experiments/v4b_tube/<run>/crops/`) + `manifest_v4b.json`
  (renamed copy of that run's `manifest.json`)

## 1. Installation


In [ ]:
# One transformers install for the ENTIRE run.
#
# 2026-07-16 -- this cell is the way it is because of a REAL failed run in which all 6 models
# died. Do not "simplify" it back:
#
#   1. transformers is upper-bounded <5. An unbounded ">=4.50" resolved to transformers
#      5.14.0, a major breaking release -- every model card in this roster was written
#      against 4.x. (5.x is what killed CheXagent-2-3b on `is_tf_available`.)
#   2. pillow / protobuf / sentencepiece are NOT installed or upgraded here. Colab already
#      ships them, and `-U pillow` left Pillow internally inconsistent (PIL/ImageText.py
#      importing `_Ink` from a PIL/_typing.py that lacks it). That breaks the torchvision
#      import chain, which transformers.image_utils imports -- so it took out 5 of the 6
#      models at once, with an error that names PIL and mentions no model at all.
#   3. No blanket `-U`. Upgrading whatever pip feels like is what caused both of the above.
#   4. torch/torchvision stay UNPINNED -- Colab ships a CUDA-linked build; forcing a
#      reinstall risks swapping in a CPU-only wheel.
!pip -q install "transformers>=4.50,<5" accelerate bitsandbytes qwen-vl-utils einops


### Environment guard

Fails in seconds if the install left a broken environment, instead of after ~20GB of model downloads and six identical tracebacks. Added after the 2026-07-16 run where a half-upgraded Pillow took out 5 of 6 models with an error that names no model at all.

In [ ]:
# Verify the environment BEFORE any model download. Each check maps to a real failure
# observed on 2026-07-16.
import PIL, transformers

# (1) transformers must be 4.x -- 5.x is a breaking release the model cards predate.
major = int(transformers.__version__.split(".")[0])
assert major == 4, (
    f"transformers is {transformers.__version__}, expected 4.x. The install cell's upper "
    "bound (<5) did not take effect -- most likely transformers was already imported in "
    "this session, so pip's change never loaded. Runtime -> Restart session, then re-run "
    "from cell 1."
)

# (2) PIL must be internally consistent AND importable through torchvision, which is what
# transformers.image_utils pulls in. A half-upgraded Pillow passes `import PIL` but dies
# here -- the exact failure that killed 5 models.
import torchvision                       # noqa: F401  -- import IS the test
from transformers import AutoProcessor   # noqa: F401  -- pulls the image_utils chain

print(f"OK | transformers {transformers.__version__} | pillow {PIL.__version__} | "
      f"torchvision imports cleanly")


## 2. Imports

In [ ]:
from __future__ import annotations

import gc
import json
import time
import traceback
import zipfile
from pathlib import Path
from typing import Callable

import torch
from PIL import Image
from tqdm.auto import tqdm

import transformers
print("transformers version:", transformers.__version__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: no GPU detected -- Runtime -> Change runtime type -> T4 GPU")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 1e9:.1f} GB total VRAM")


### Hugging Face login (required for gated `google/medgemma-4b-it`)

Accept the license at https://huggingface.co/google/medgemma-4b-it first, then paste a read-scoped token below. If you skip this, every OTHER model still runs -- only medgemma will fail-and-skip with an auth error.

In [ ]:
from huggingface_hub import login
try:
    login()  # opens a token prompt; paste a token with at least 'read' scope
except Exception as e:
    print(f"HF login skipped/failed ({e!r}) -- medgemma will fail-and-skip later, "
          "everything else is unaffected")


## 3. Upload Dataset

Upload all four files in one picker: `crops_v4a.zip`, `manifest_v4a.json`, `crops_v4b.zip`, `manifest_v4b.json`.

In [ ]:
from google.colab import files

print("Upload crops_v4a.zip, manifest_v4a.json, crops_v4b.zip, manifest_v4b.json ...")
uploaded = files.upload()

REQUIRED = ["crops_v4a.zip", "manifest_v4a.json", "crops_v4b.zip", "manifest_v4b.json"]
missing = [f for f in REQUIRED if f not in uploaded]
assert not missing, f"missing required upload(s): {missing}"
print("all 4 files present")


## 4. Extract Dataset

In [ ]:
def extract_crops(zip_name: str, dest_dir: str) -> None:
    """Extract a crops zip into dest_dir, tolerating a nested 'crops/' folder inside it."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(dest)
    # Some zips contain crops/*.png directly, others nest an extra crops/ level.
    nested = dest / "crops"
    if nested.is_dir() and not any(dest.glob("*.png")):
        for p in nested.iterdir():
            p.rename(dest / p.name)
        nested.rmdir()
    n = len(list(dest.glob("*.png")))
    assert n > 0, f"no .png crops found in {dest_dir} after extracting {zip_name}"
    print(f"{dest_dir}: {n} crops")

extract_crops("crops_v4a.zip", "crops_v4a")
extract_crops("crops_v4b.zip", "crops_v4b")


## 5. Read Manifest

In [ ]:
def load_manifest(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        manifest = json.load(f)
    n_pos = sum(1 for r in manifest if r["kind"] == "positive")
    n_ctrl = sum(1 for r in manifest if r["kind"] == "control")
    print(f"{path}: {len(manifest)} rows ({n_pos} positive, {n_ctrl} control)")
    return manifest

DATASETS: dict[str, dict] = {
    "v4a": {"manifest": load_manifest("manifest_v4a.json"), "crops_dir": "crops_v4a"},
    "v4b": {"manifest": load_manifest("manifest_v4b.json"), "crops_dir": "crops_v4b"},
}


## 6. Prompt

Byte-identical across every model and every dataset -- the only independent variable is the model.

In [ ]:
PROMPT = (
    "This is a small region that a diagnostic model focused on within a chest X-ray. "
    "Describe anything artificial, non-anatomical, or out of place that you see in this "
    "region. If everything looks like normal anatomy, say so. Be specific and concise."
)
MAX_NEW_TOKENS = 64


## 7. Benchmark Functions

Each model family gets a **loader** (`(model_id, quantize) -> (processor_or_tokenizer, model)`)
and a **generate function** (`(processor_or_tokenizer, model, image_path, prompt) -> str`).
A generic `AutoProcessor` + `AutoModelForImageTextToText` + chat-template path covers most of
the roster; CheXagent and Phi-3.5-vision use their own documented `trust_remote_code=True`
calling conventions and get dedicated functions.

In [ ]:
def make_bnb_config() -> "transformers.BitsAndBytesConfig":
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )


def vram_report(label: str) -> None:
    """Print currently allocated/reserved VRAM -- called before every model load."""
    if DEVICE != "cuda":
        return
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"[{label}] VRAM allocated={allocated:.2f} GB reserved={reserved:.2f} GB")


def unload(*objs) -> None:
    """Fully release a model/processor pair and clear CUDA VRAM before the next model."""
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [ ]:
# --- Generic family: AutoProcessor + AutoModelForImageTextToText -----------------------
# Covers Qwen2.5-VL, MedGemma, SmolVLM2 -- all documented (as of their current model cards) to
# load via the generic chat-template Vision2Seq-successor class and a {"role": "user",
# "content": [{"type": "image", ...}, {"type": "text", ...}]} message.

def load_generic_itt(model_id: str, quantize: bool):
    from transformers import AutoModelForImageTextToText, AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    kwargs = dict(device_map="auto", trust_remote_code=True)
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
            return processor, model
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to bf16")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = torch.bfloat16
    model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    return processor, model


def generate_generic_itt(processor, model, image_path: str, prompt: str) -> str:
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                              {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- Qwen2.5-VL: its OWN model class + qwen-vl-utils builds the vision inputs ----------
# Verified against the live model card (2026-07-16): the card loads
# Qwen2_5_VLForConditionalGeneration explicitly, not the generic AutoModelForImageTextToText
# used by the other three -- using the specific class here rather than assuming the generic
# Auto class resolves the same config mapping.

def load_qwen25vl(model_id: str, quantize: bool):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(model_id)
    kwargs = dict(device_map="auto")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)
            return processor, model
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to bf16")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = torch.bfloat16
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)
    return processor, model


def generate_qwen25vl(processor, model, image_path: str, prompt: str) -> str:
    from qwen_vl_utils import process_vision_info
    messages = [{"role": "user", "content": [{"type": "image", "image": image_path},
                                              {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                        padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- Phi-3.5-vision-instruct: AutoModelForCausalLM + trust_remote_code, own <|image_1|> tag ---
# Predates the generic ImageTextToText class; this is the model card's own documented pattern.

def load_phi35_vision(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, num_crops=4)
    kwargs = dict(device_map="auto", trust_remote_code=True, _attn_implementation="eager")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
            return processor, model
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to bf16")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return processor, model


def generate_phi35_vision(processor, model, image_path: str, prompt: str) -> str:
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": f"<|image_1|>\n{prompt}"}]
    text = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, [image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- CheXagent-2-3b: AutoModelForCausalLM + AutoTokenizer, from_list_format ------------
# Verbatim pattern verified against the LIVE model card on 2026-07-16 (also matches
# src/auditor/discovery/colab/chexagent_naming.ipynb, checked against the same card on
# 2026-07-15). NOT quantized -- trust_remote_code architectures often don't support
# bitsandbytes cleanly, and this family is already the highest transformers-version risk
# (see markdown intro).

def load_chexagent_2_3b(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto",
                                                  trust_remote_code=True)
    model = model.to(torch.bfloat16)
    model.eval()
    return tokenizer, model


def generate_chexagent_2_3b(tokenizer, model, image_path: str, prompt: str) -> str:
    query = tokenizer.from_list_format([{"image": image_path}, {"text": prompt}])
    conv = [{"from": "system", "value": "You are a helpful assistant."},
            {"from": "human", "value": query}]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True,
                                               return_tensors="pt")
    output = model.generate(
        input_ids.to(model.device), do_sample=False, num_beams=1, temperature=1.0, top_p=1.0,
        use_cache=True, max_new_tokens=MAX_NEW_TOKENS,
    )[0]
    return tokenizer.decode(output[input_ids.size(1):-1]).strip()


# --- CheXagent-8b: a DIFFERENT convention from the 3b model -- AutoProcessor (not
# AutoTokenizer/from_list_format) + a model-supplied GenerationConfig. Verified against the
# live model card on 2026-07-16; initially assumed (wrongly) to share the 3b's calling
# convention -- confirmed via WebFetch that it does not, so it gets its own functions rather
# than reusing load_chexagent_2_3b.
#
# UNLIKE the 3b model, this one IS quantized by default: 8B params at fp16 is ~16GB of
# weights alone, which does not leave headroom for activations on the T4 (15.6GB usable, per
# the 2026-07-16 run's own GPU probe) and would OOM. 4-bit is attempted first with an fp16
# fallback, same pattern as the other quantized families.
#
# NOTE: this model has never actually reached the memory question -- on 2026-07-16 it died at
# import on the broken-Pillow bug, well before allocating anything. So whether 4-bit works on
# this trust_remote_code architecture is UNTESTED, not established. If it fails, that logs
# FAILED and the run continues, per the skip-and-log policy.

def load_chexagent_8b(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    generation_config = GenerationConfig.from_pretrained(model_id)
    kwargs = dict(trust_remote_code=True, device_map="auto")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
            return (processor, generation_config), model
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to fp16 (likely to OOM on a T4)")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return (processor, generation_config), model


def generate_chexagent_8b(proc_and_genconfig, model, image_path: str, prompt: str) -> str:
    processor, generation_config = proc_and_genconfig
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=[image], text=f" USER: <s>{prompt} ASSISTANT: <s>",
                        return_tensors="pt").to(device=model.device, dtype=torch.float16)
    output = model.generate(**inputs, generation_config=generation_config)[0]
    return processor.tokenizer.decode(output, skip_special_tokens=True).strip()


In [ ]:
# --- Roster: (name, model_id, quantize, loader, generate_fn) ---------------------------
ROSTER: list[dict] = [
    {"name": "chexagent_2_3b", "model_id": "StanfordAIMI/CheXagent-2-3b", "quantize": False,
     "load": load_chexagent_2_3b, "generate": generate_chexagent_2_3b},
    {"name": "chexagent_8b", "model_id": "StanfordAIMI/CheXagent-8b", "quantize": True,
     "load": load_chexagent_8b, "generate": generate_chexagent_8b},
    {"name": "medgemma_4b_it", "model_id": "google/medgemma-4b-it", "quantize": True,
     "load": load_generic_itt, "generate": generate_generic_itt},
    {"name": "qwen25_vl_3b", "model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "quantize": True,
     "load": load_qwen25vl, "generate": generate_qwen25vl},
    {"name": "phi35_vision", "model_id": "microsoft/Phi-3.5-vision-instruct", "quantize": True,
     "load": load_phi35_vision, "generate": generate_phi35_vision},
    {"name": "smolvlm2_2_2b", "model_id": "HuggingFaceTB/SmolVLM2-2.2B-Instruct", "quantize": True,
     "load": load_generic_itt, "generate": generate_generic_itt},
]
print(f"roster: {len(ROSTER)} models")


In [ ]:
def run_one_dataset(generate_fn: Callable, model, proc_or_tok, dataset_name: str,
                    manifest: list[dict], crops_dir: str, out_path: Path) -> dict:
    """Run PROMPT over every crop in one dataset, write predictions_<model>_<dataset>.jsonl.

    Returns a small timing summary dict; never raises on a single-image failure -- a failed
    image is skipped (not written) and counted, so one bad crop can't kill the whole model run.
    """
    n_ok = n_failed = 0
    t0 = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for row in tqdm(manifest, desc=f"{dataset_name}", leave=False):
            image_path = f"{crops_dir}/{row['id']}.png"
            try:
                desc = generate_fn(proc_or_tok, model, image_path, PROMPT)
                n_ok += 1
            except Exception as e:
                desc = ""
                n_failed += 1
                print(f"  IMAGE FAILED {row['id']}: {e!r}")
            f.write(json.dumps({"id": row["id"], "kind": row["kind"], "description": desc}) + "\n")
    elapsed = time.time() - t0
    n_total = n_ok + n_failed
    rate = n_total / elapsed if elapsed > 0 else 0.0
    print(f"  {dataset_name}: {n_ok}/{n_total} ok, {elapsed:.1f}s total, "
          f"{rate:.2f} img/s, {elapsed / max(n_total, 1):.2f} s/img avg -> {out_path}")
    return {"dataset": dataset_name, "n_ok": n_ok, "n_failed": n_failed,
            "elapsed_s": round(elapsed, 1), "images_per_sec": round(rate, 3)}


## 8. Benchmark Loop

Loads each model once, runs it over V4a then V4b, unloads, moves on. Never stops the whole
benchmark on a single model's failure -- OOM / unsupported architecture / tokenizer issues are
caught, logged to `run_log.md`, and the next model runs. **No metrics are computed here** --
this notebook's job ends at `predictions_*.jsonl`; see the intro markdown cell for the local
scoring command (reuses this project's existing tested matchers).

In [ ]:
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

run_log: list[dict] = []
overall_t0 = time.time()

for spec in ROSTER:
    name, model_id = spec["name"], spec["model_id"]
    print(f"\n{'=' * 70}\n{name} ({model_id})\n{'=' * 70}")
    vram_report(f"before {name}")

    model = proc_or_tok = None
    try:
        proc_or_tok, model = spec["load"](model_id, spec["quantize"])
        print(f"  loaded ok")
    except Exception as e:
        reason = f"{type(e).__name__}: {e}"
        print(f"  FAILED to load -- {reason}")
        traceback.print_exc()
        run_log.append({"model": name, "model_id": model_id, "status": "FAILED_LOAD",
                         "reason": reason})
        unload(model, proc_or_tok)
        continue

    model_results = []
    try:
        for dataset_name, d in DATASETS.items():
            out_path = OUT_DIR / f"predictions_{name}_{dataset_name}.jsonl"
            result = run_one_dataset(spec["generate"], model, proc_or_tok, dataset_name,
                                      d["manifest"], d["crops_dir"], out_path)
            model_results.append(result)
        run_log.append({"model": name, "model_id": model_id, "status": "OK",
                         "results": model_results})
    except Exception as e:
        reason = f"{type(e).__name__}: {e}"
        print(f"  FAILED during inference -- {reason}")
        traceback.print_exc()
        run_log.append({"model": name, "model_id": model_id, "status": "FAILED_INFERENCE",
                         "reason": reason, "results": model_results})
    finally:
        unload(model, proc_or_tok)
        vram_report(f"after {name}")

print(f"\nTOTAL benchmark runtime: {time.time() - overall_t0:.1f}s")


## 9. Results

In [ ]:
for entry in run_log:
    print(f"{entry['model']:20s} {entry['status']}")
    if entry["status"] == "OK":
        for r in entry["results"]:
            print(f"    {r['dataset']}: {r['n_ok']} ok / {r['n_failed']} failed, "
                  f"{r['images_per_sec']:.2f} img/s")
    else:
        print(f"    reason: {entry['reason']}")


## 10. Summary

In [ ]:
summary_lines = ["# Open-weight VLM roster benchmark -- run log\n"]
for entry in run_log:
    summary_lines.append(f"## {entry['model']} (`{entry['model_id']}`)\n")
    summary_lines.append(f"Status: **{entry['status']}**\n")
    if entry["status"] == "OK":
        for r in entry["results"]:
            summary_lines.append(
                f"- {r['dataset']}: {r['n_ok']}/{r['n_ok'] + r['n_failed']} images ok, "
                f"{r['elapsed_s']}s, {r['images_per_sec']} img/s\n"
            )
    else:
        summary_lines.append(f"- reason: {entry['reason']}\n")
    summary_lines.append("\n")

(OUT_DIR / "run_log.md").write_text("".join(summary_lines), encoding="utf-8")
(OUT_DIR / "run_log.json").write_text(json.dumps(run_log, indent=2), encoding="utf-8")
print("".join(summary_lines))


In [ ]:
# Zip everything for one-click download, then score locally (see intro markdown cell).
import shutil
shutil.make_archive("outputs", "zip", OUT_DIR)

from google.colab import files
files.download("outputs.zip")
